# Identification checks: lead placebo, 2018+ sample, multiple testing

Three checks on the main reduced-form and IV results:

1. **Lead (placebo-in-time) test.** Next month's upwind fire cannot cause this month's conflict. If it predicts conflict, the instrument is picking up seasonal or regional trends rather than smoke.
2. **2018–2025 sample.** ACLED's Indonesia coverage roughly triples in 2018 (333 events in 2017, 1,092 in 2018). Re-estimating on 2018 onward checks that the results do not depend on the thinly covered early years.
3. **Multiple-testing correction.** Four outcomes are tested. Sharpened q-values (Anderson 2008, following Benjamini, Krieger and Yekutieli 2006) control the false discovery rate across them.

Specifications follow the paper: district and year-month fixed effects, standard errors clustered by province, conflict-active sample = districts with any event in at least 30% of months over 2015–2025. Reads `iv_panel_district.csv` from `wind-IV.ipynb`.

In [1]:
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import pyfixest as pf
from scipy import stats

warnings.filterwarnings('ignore')

ROOT         = '/Users/hariaksha/Documents/GitHub/climate-conflict'
PANEL_CSV    = f'{ROOT}/analysis/iv_panel_district.csv'
DISTRICT_SHP = f'{ROOT}/data/administrative/gadm41_IDN_shp/gadm41_IDN_2.shp'
ACLED_XLSX   = f'{ROOT}/data/unrest/ACLED Data_2026-06-10.xlsx'
OUT_CSV      = f'{ROOT}/analysis/output/identification_checks_results.csv'

INSTR    = ['log_upwind_frp', 'log_upwind_frp_l1']
OUTCOMES = ['events', 'pv_events', 'riots_vac', 'vac']
FE       = 'district + year_month'
VCOV     = {'CRV1': 'province'}

panel = pd.read_csv(PANEL_CSV)
print(f'Panel: {len(panel):,} obs, {panel["district"].nunique()} districts')

Panel: 60,125 obs, 481 districts


## 1. Event-type outcomes

Assigned to districts by coordinates exactly as in `weather-controls.ipynb`, Section 4.

In [2]:
districts_gdf = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(
    columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = districts_gdf['district'].duplicated(keep=False)
districts_gdf.loc[dup, 'district'] = (districts_gdf.loc[dup, 'district'] + ' ('
                                      + districts_gdf.loc[dup, 'province'] + ')')

acled = pd.read_excel(ACLED_XLSX, usecols=['event_date', 'event_type', 'latitude', 'longitude'])
acled['event_date'] = pd.to_datetime(acled['event_date'])
acled['year'], acled['month'] = acled['event_date'].dt.year, acled['event_date'].dt.month
ag = gpd.GeoDataFrame(acled, geometry=gpd.points_from_xy(acled['longitude'], acled['latitude']), crs='EPSG:4326')
j = gpd.sjoin(ag, districts_gdf[['district', 'geometry']], how='left', predicate='within')
j = j[~j.index.duplicated(keep='first')]
acled['district'] = j['district'].reindex(acled.index).values
off = acled['district'].isna()
near = gpd.sjoin_nearest(ag.loc[off].to_crs(3395), districts_gdf[['district', 'geometry']].to_crs(3395),
                         how='left', max_distance=20_000)
near = near[~near.index.duplicated(keep='first')]
acled.loc[off, 'district'] = near['district'].reindex(acled.index[off]).values

acled['riots_vac'] = acled['event_type'].isin(['Riots', 'Violence against civilians']).astype(int)
acled['vac'] = (acled['event_type'] == 'Violence against civilians').astype(int)
counts = acled.groupby(['district', 'year', 'month'])[['riots_vac', 'vac']].sum().reset_index()
panel = panel.merge(counts, on=['district', 'year', 'month'], how='left').fillna({'riots_vac': 0, 'vac': 0})
print(panel[OUTCOMES].sum())

events       21653.0
pv_events     3520.0
riots_vac     2824.0
vac           1067.0
dtype: float64


## 2. Helpers and samples

`effective_f` is the Montiel Olea–Pflueger effective F, and `ar_p` the Anderson–Rubin p-value, both as in `weather-controls.ipynb`. The lead of upwind FRP is built within district; the last month of each district's panel has no lead and drops from specifications that include it.

In [3]:
def effective_f(data, instruments, controls=(), fe=FE):
    data = data.reset_index(drop=True)
    ctrl = ' + '.join(controls)
    rhs = ' + '.join(list(instruments) + list(controls))
    fs = pf.feols(f'log_frp ~ {rhs} | {fe}', data=data, vcov=VCOV)
    Z = np.column_stack([pf.feols(f'{z} ~ {ctrl if ctrl else "1"} | {fe}', data=data).resid()
                         for z in instruments])
    names = list(fs.coef().index)
    idx = [names.index(z) for z in instruments]
    pi = fs.coef()[list(instruments)].to_numpy()
    V = fs._vcov[np.ix_(idx, idx)]
    S = Z.T @ Z
    return float(pi @ S @ pi / np.trace(V @ S))

def wald_p(m, terms, G):
    names = list(m.coef().index)
    idx = [names.index(t) for t in terms]
    b = m.coef()[terms].to_numpy()
    V = m._vcov[np.ix_(idx, idx)]
    W = float(b @ np.linalg.solve(V, b))
    return stats.f.sf(W / len(terms), len(terms), G - 1)

def ar_p(data, y, controls=('log_frp_l1',), fe=FE):
    m = pf.feols(f'{y} ~ {" + ".join(INSTR + list(controls))} | {fe}', data=data, vcov=VCOV)
    return wald_p(m, INSTR, data['province'].nunique())

panel = panel.sort_values(['district', 'year', 'month'])
panel['log_upwind_frp_f1'] = panel.groupby('district')['log_upwind_frp'].shift(-1)
panel['log_frp_f1'] = panel.groupby('district')['log_frp'].shift(-1)

share = panel.groupby('district')['events'].apply(lambda s: (s > 0).mean())
active = share[share >= 0.30].index
samples = {'full panel': panel, 'active >= 30%': panel[panel['district'].isin(active)]}
print(f'Active >= 30% districts: {len(active)}')

rows = []

Active >= 30% districts: 85


## 3. Lead (placebo-in-time) test

Reduced form with the lead added: $y_{dt}$ on $Z_{d,t+1}$, $Z_{dt}$, $Z_{d,t-1}$. The lead should be small and insignificant, and adding it should leave the contemporaneous coefficient unchanged. Two variants: the baseline reduced form estimated on the same observations (those with a lead), and the reduced form with the lead included. As a second version, the lead of the district's own fire (log FRP) is also added as a control, since next month's local fire partly reflects this month's conditions.

In [4]:
for sample, d in samples.items():
    d = d.dropna(subset=['log_upwind_frp_f1', 'log_frp_f1'])
    G = d['province'].nunique()
    print(f'\n== {sample} ({len(d):,} obs) ==')
    for y in OUTCOMES:
        base = pf.feols(f'{y} ~ {" + ".join(INSTR)} | {FE}', data=d, vcov=VCOV).tidy()
        lead = pf.feols(f'{y} ~ log_upwind_frp_f1 + {" + ".join(INSTR)} | {FE}', data=d, vcov=VCOV).tidy()
        lead2 = pf.feols(f'{y} ~ log_upwind_frp_f1 + log_frp_f1 + {" + ".join(INSTR)} | {FE}',
                         data=d, vcov=VCOV).tidy()
        r = {'check': 'lead test', 'sample': sample, 'outcome': y,
             'rf_contemp_baseline': base.loc['log_upwind_frp', 'Estimate'],
             'rf_contemp_baseline_p': base.loc['log_upwind_frp', 'Pr(>|t|)'],
             'lead_coef': lead.loc['log_upwind_frp_f1', 'Estimate'],
             'lead_p': lead.loc['log_upwind_frp_f1', 'Pr(>|t|)'],
             'rf_contemp_with_lead': lead.loc['log_upwind_frp', 'Estimate'],
             'rf_contemp_with_lead_p': lead.loc['log_upwind_frp', 'Pr(>|t|)'],
             'lead_coef_ownfire_ctrl': lead2.loc['log_upwind_frp_f1', 'Estimate'],
             'lead_p_ownfire_ctrl': lead2.loc['log_upwind_frp_f1', 'Pr(>|t|)']}
        rows.append(r)
        print(f"  {y:10s} lead {r['lead_coef']:+.4f} (p={r['lead_p']:.3f}) | contemp. {r['rf_contemp_baseline']:+.4f} "
              f"(p={r['rf_contemp_baseline_p']:.3f}) -> {r['rf_contemp_with_lead']:+.4f} (p={r['rf_contemp_with_lead_p']:.3f}) "
              f"| lead w/ own-fire lead ctrl {r['lead_coef_ownfire_ctrl']:+.4f} (p={r['lead_p_ownfire_ctrl']:.3f})")


== full panel (59,644 obs) ==


  events     lead +0.0012 (p=0.693) | contemp. -0.0023 (p=0.268) -> -0.0028 (p=0.134) | lead w/ own-fire lead ctrl +0.0011 (p=0.712)


  pv_events  lead -0.0008 (p=0.429) | contemp. -0.0007 (p=0.222) -> -0.0004 (p=0.393) | lead w/ own-fire lead ctrl -0.0007 (p=0.488)


  riots_vac  lead -0.0007 (p=0.452) | contemp. -0.0008 (p=0.198) -> -0.0005 (p=0.325) | lead w/ own-fire lead ctrl -0.0004 (p=0.596)


  vac        lead +0.0002 (p=0.543) | contemp. -0.0007 (p=0.153) -> -0.0008 (p=0.107) | lead w/ own-fire lead ctrl +0.0003 (p=0.236)

== active >= 30% (10,540 obs) ==
  events     lead -0.0002 (p=0.985) | contemp. -0.0257 (p=0.003) -> -0.0256 (p=0.001) | lead w/ own-fire lead ctrl +0.0014 (p=0.905)
  pv_events  lead -0.0024 (p=0.250) | contemp. -0.0071 (p=0.078) -> -0.0061 (p=0.122) | lead w/ own-fire lead ctrl -0.0014 (p=0.532)
  riots_vac  lead -0.0019 (p=0.328) | contemp. -0.0078 (p=0.037) -> -0.0071 (p=0.070) | lead w/ own-fire lead ctrl -0.0008 (p=0.699)


  vac        lead +0.0008 (p=0.609) | contemp. -0.0055 (p=0.066) -> -0.0058 (p=0.085) | lead w/ own-fire lead ctrl +0.0012 (p=0.510)


## 4. 2018–2025 sample

Main specifications re-estimated on 2018 onward: reduced form, distributed-lag IV with lagged log FRP as a control, effective F and Anderson–Rubin p-value. The conflict-active sample is shown two ways: the same 85 districts as in the paper, and districts re-selected using their 2018–2025 event share.

In [5]:
late = panel[panel['year'] >= 2018]
share_late = late.groupby('district')['events'].apply(lambda s: (s > 0).mean())
active_late = share_late[share_late >= 0.30].index
late_samples = {
    'full panel, 2018+': late,
    'active >= 30% (paper districts), 2018+': late[late['district'].isin(active)],
    'active >= 30% (re-selected on 2018+), 2018+': late[late['district'].isin(active_late)],
}
print(f'Districts active >= 30% on 2018-2025 shares: {len(active_late)} '
      f'(overlap with paper\'s {len(active)}: {len(set(active) & set(active_late))})')

for sample, d in late_samples.items():
    effF = effective_f(d, INSTR, ['log_frp_l1'])
    print(f'\n== {sample} ({d["district"].nunique()} districts, {len(d):,} obs) | effective F = {effF:.1f} ==')
    for y in OUTCOMES:
        rf = pf.feols(f'{y} ~ {" + ".join(INSTR)} | {FE}', data=d, vcov=VCOV).tidy().loc['log_upwind_frp']
        iv = pf.feols(f'{y} ~ log_frp_l1 | {FE} | log_frp ~ {" + ".join(INSTR)}',
                      data=d, vcov=VCOV).tidy().loc['log_frp']
        arp = ar_p(d, y)
        rows.append({'check': '2018+ sample', 'sample': sample, 'outcome': y, 'effective_F': effF,
                     'rf_coef': rf['Estimate'], 'rf_p': rf['Pr(>|t|)'],
                     'iv_coef': iv['Estimate'], 'iv_p': iv['Pr(>|t|)'], 'ar_p': arp})
        print(f"  {y:10s} RF {rf['Estimate']:+.4f} (p={rf['Pr(>|t|)']:.3f}) | IV {iv['Estimate']:+.4f} "
              f"(p={iv['Pr(>|t|)']:.3f}) | AR p={arp:.3f}")

Districts active >= 30% on 2018-2025 shares: 117 (overlap with paper's 85: 85)

== full panel, 2018+ (481 districts, 43,290 obs) | effective F = 69.2 ==


  events     RF -0.0059 (p=0.079) | IV -0.0188 (p=0.215) | AR p=0.048


  pv_events  RF -0.0013 (p=0.302) | IV -0.0036 (p=0.467) | AR p=0.007


  riots_vac  RF -0.0009 (p=0.327) | IV -0.0019 (p=0.601) | AR p=0.006


  vac        RF -0.0010 (p=0.204) | IV -0.0036 (p=0.230) | AR p=0.024

== active >= 30% (paper districts), 2018+ (85 districts, 7,650 obs) | effective F = 25.5 ==
  events     RF -0.0362 (p=0.003) | IV -0.2511 (p=0.001) | AR p=0.015
  pv_events  RF -0.0107 (p=0.097) | IV -0.0716 (p=0.055) | AR p=0.079


  riots_vac  RF -0.0097 (p=0.050) | IV -0.0631 (p=0.025) | AR p=0.034
  vac        RF -0.0068 (p=0.093) | IV -0.0463 (p=0.039) | AR p=0.011

== active >= 30% (re-selected on 2018+), 2018+ (117 districts, 10,530 obs) | effective F = 35.4 ==
  events     RF -0.0303 (p=0.002) | IV -0.1748 (p=0.001) | AR p=0.017


  pv_events  RF -0.0105 (p=0.055) | IV -0.0554 (p=0.039) | AR p=0.045
  riots_vac  RF -0.0097 (p=0.031) | IV -0.0488 (p=0.028) | AR p=0.004
  vac        RF -0.0055 (p=0.121) | IV -0.0277 (p=0.116) | AR p=0.010


## 5. Multiple-testing correction across the four outcomes

Sharpened q-values (Anderson 2008, Section 3.3; the two-stage procedure of Benjamini, Krieger and Yekutieli 2006). A q-value is the smallest false discovery rate at which the hypothesis is rejected. They are computed within each family of four outcomes, for three families in the conflict-active sample: the reduced-form contemporaneous coefficient, the 2SLS coefficient and the Anderson–Rubin test. (Bonferroni-adjusted p-values, a stricter family-wise correction, are shown for comparison.)

In [6]:
def sharpened_q(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    q = np.ones(m)
    for qq in np.arange(1.0, 0.0, -0.001):
        q1 = qq / (1 + qq)
        ranks = np.arange(1, m + 1)
        rej = p[order] <= ranks * q1 / m
        r1 = ranks[rej].max() if rej.any() else 0
        q2 = q1 * m / (m - r1)
        rej2 = p[order] <= ranks * q2 / m
        r2 = ranks[rej2].max() if rej2.any() else 0
        q[order[:r2]] = qq
    return q

d = samples['active >= 30%']
fam = {'reduced form': [], '2SLS': [], 'Anderson-Rubin': []}
for y in OUTCOMES:
    fam['reduced form'].append(pf.feols(f'{y} ~ {" + ".join(INSTR)} | {FE}', data=d, vcov=VCOV)
                               .tidy().loc['log_upwind_frp', 'Pr(>|t|)'])
    fam['2SLS'].append(pf.feols(f'{y} ~ log_frp_l1 | {FE} | log_frp ~ {" + ".join(INSTR)}', data=d, vcov=VCOV)
                       .tidy().loc['log_frp', 'Pr(>|t|)'])
    fam['Anderson-Rubin'].append(ar_p(d, y))

for name, ps in fam.items():
    q = sharpened_q(ps)
    print(f'\n{name} (active >= 30%):')
    for y, p, qq in zip(OUTCOMES, ps, q):
        print(f'  {y:10s} p = {p:.3f} | sharpened q = {qq:.3f} | Bonferroni = {min(1, 4 * p):.3f}')
        rows.append({'check': 'multiple testing', 'sample': 'active >= 30%', 'outcome': y,
                     'family': name, 'p': p, 'sharpened_q': qq, 'bonferroni_p': min(1, 4 * p)})

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV}')


reduced form (active >= 30%):
  events     p = 0.002 | sharpened q = 0.009 | Bonferroni = 0.009
  pv_events  p = 0.078 | sharpened q = 0.063 | Bonferroni = 0.313
  riots_vac  p = 0.039 | sharpened q = 0.063 | Bonferroni = 0.156
  vac        p = 0.068 | sharpened q = 0.063 | Bonferroni = 0.274

2SLS (active >= 30%):
  events     p = 0.000 | sharpened q = 0.002 | Bonferroni = 0.001
  pv_events  p = 0.042 | sharpened q = 0.033 | Bonferroni = 0.167
  riots_vac  p = 0.017 | sharpened q = 0.026 | Bonferroni = 0.066
  vac        p = 0.026 | sharpened q = 0.027 | Bonferroni = 0.103

Anderson-Rubin (active >= 30%):
  events     p = 0.012 | sharpened q = 0.050 | Bonferroni = 0.047
  pv_events  p = 0.164 | sharpened q = 0.090 | Bonferroni = 0.655
  riots_vac  p = 0.066 | sharpened q = 0.050 | Bonferroni = 0.263
  vac        p = 0.024 | sharpened q = 0.050 | Bonferroni = 0.095

Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/identification_checks_results.csv
